# Dichroic multi-state phase retrieval
Minimal use of `phase_retrieval_core_dichroic.py`.

In [ ]:
import numpy as np
from library import phase_retrieval_core_dichroic as pr

In [ ]:
holograms = np.load("data/dichroic_holograms.npy")  # (n_observations, nx, ny)
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")
state_labels = ["saturated", "saturated", "domains", "domains"]
polarization_signs = [+1, -1, +1, -1]
saturated_states = {"saturated": +1}

In [ ]:
recipe = {
    # Independent phase-retrieval updates for every observation.
    "inner_mode": ["HAPRE", "ER"],       # Algorithms repeated in each outer cycle.
    "inner_Nit": [700, 50],               # Iterations in each inner stage.
    "outer_iterations": 100,              # Number of update-plus-dichroic-projection cycles.
    "warmup_mode": ["HAPRE"],            # Independent algorithms before joint fitting.
    "warmup_Nit": 0,                      # Zero disables warmup.
    "shuffle_observations": True,         # Randomize observation order each cycle.
    "random_seed": None,                  # Seed for observation-order randomization.
    "beta_zero": 0.5,                     # Beta value(s), scalar or one per stage.
    "beta_mode": "arctan",               # Beta schedule name(s) or arrays.
    "alpha_zero": 0.0,                    # TV strength; zero disables TV.
    "alpha_mode": "const",               # Alpha schedule name(s) or arrays.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits the inner setting.
    "warmup_beta_mode": None,             # None inherits the inner setting.
    "warmup_alpha_zero": None,            # None inherits the inner setting.
    "warmup_alpha_mode": None,            # None inherits the inner setting.
    "warmup_TV_freq": None,               # None inherits the inner setting.
    "plot_every": 1e9,                    # Error sampling/plot interval.
    "average_img": 1,                     # Number of best late iterates to average.
    "Fourier_last": True,                 # Finish every stage with its Fourier constraint.
    "final_fourier_constraint": True,     # Finish outputs on measured amplitudes.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction.
    # Dichroic object model.
    "projection_model": "saturated_reference", # shared_charge, saturated_reference, or none.
    "projection_every": 1,                # Joint-projection interval in outer cycles.
    "projection_start": 0,                # First cycle eligible for projection.
    "projection_relaxation": 1.0,         # Projection blending fraction.
    "observation_weights": None,          # Positive weight per hologram.
    "rank_deficient": "error",           # "error" or minimum-norm handling for weak designs.
    "saturated_states": saturated_states, # State-to-+1/-1 mapping; None means no saturation prior.
    "clip_magnetization": True,           # Always constrain reduced magnetization to [-1, +1].
    "kt_delta_m_range": None,             # Optional bounds on k*t*delta_m.
    "kt_beta_m_range": None,              # Optional bounds on k*t*beta_m.
    "log_floor": 1e-12,                   # Magnitude floor before complex logarithm.
}

fields, components, bsmasks, errors = (
    pr.dichroic_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        state_labels=state_labels,
        polarization_signs=polarization_signs,
        saturated_states=saturated_states,
        dichroic_recipe=recipe,
    )
)